# Week 11 Pandas and DuckDB Examples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST2412_Data_Security_Privacy_Ethics/blob/main/week_11/week11_pandas_duckdb_examples.ipynb)

Companion notebook for Week 11.
This notebook mirrors the Day 1 and Day 2 analytics logic used in the Week 11 labs.

What it includes:
- `pandas` examples for Day 1 and Day 2
- `DuckDB` SQL examples for Day 1 and Day 2
- GitHub raw URLs so the notebook can pull the current Week 11 CSVs from the repository

Files used:
- `week_11/data/day1_auth_events.csv`
- `week_11/data/day1_user_context.csv`
- `week_11/data/day2_alert_queue.csv`
- `week_11/data/day2_asset_context.csv`


In [ ]:
# If needed, uncomment this cell and run it once.
# %pip install pandas duckdb


In [ ]:
import pandas as pd
import duckdb

BASE_URL = 'https://github.com/lolusername/CST2412_Data_Security_Privacy_Ethics/raw/main/week_11/data'
DAY1_EVENTS = f"{BASE_URL}/day1_auth_events.csv"
DAY1_USERS = f"{BASE_URL}/day1_user_context.csv"
DAY2_ALERTS = f"{BASE_URL}/day2_alert_queue.csv"
DAY2_ASSETS = f"{BASE_URL}/day2_asset_context.csv"

print('Base URL:', BASE_URL)
print('Day 1 events:', DAY1_EVENTS)
print('Day 2 alerts:', DAY2_ALERTS)


## Day 1 with Pandas

Goal:
- inspect authentication activity
- count failures by source IP
- identify the likely password-spray source
- enrich the suspicious activity with account context


In [ ]:
events = pd.read_csv(DAY1_EVENTS, parse_dates=['timestamp'])
users = pd.read_csv(DAY1_USERS)

print('Total rows:', len(events))
print('Event types:')
print(events['event_type'].value_counts())


In [ ]:
failed = events.query("event_type == 'login_failed'")
failed_by_ip = (
    failed.groupby('src_ip')
    .agg(
        failed_count=('event_id', 'count'),
        targeted_users=('user_id', 'nunique')
    )
    .sort_values('failed_count', ascending=False)
)
failed_by_ip


In [ ]:
spray_ip = failed_by_ip.index[0]
print('Likely suspicious source IP:', spray_ip)

failed.loc[
    failed['src_ip'] == spray_ip,
    ['timestamp', 'user_id', 'device', 'city']
].sort_values('timestamp')


In [ ]:
enriched_events = events.merge(users, on='user_id', how='left')

enriched_events.loc[
    (enriched_events['src_ip'] == spray_ip) | (enriched_events['user_id'] == 'u_admin_fin'),
    ['timestamp', 'src_ip', 'user_id', 'event_type', 'outcome', 'privileged_account', 'sensitive_account', 'department', 'role_title']
].sort_values(['timestamp', 'user_id'])


In [ ]:
u_admin_fin_sequence = enriched_events.loc[
    enriched_events['user_id'] == 'u_admin_fin',
    ['timestamp', 'src_ip', 'event_type', 'outcome', 'factor', 'privileged_account', 'sensitive_account']
].sort_values('timestamp')

u_admin_fin_sequence


## Day 1 with DuckDB SQL

DuckDB is a good fit here because the Week 11 data already lives in CSV files.
This lets you show SQL reasoning without setting up a separate database server.


In [ ]:
con = duckdb.connect()

day1_count_query = f"""
SELECT
    src_ip,
    COUNT(*) AS failed_count,
    COUNT(DISTINCT user_id) AS targeted_users
FROM read_csv_auto('{DAY1_EVENTS}')
WHERE event_type = 'login_failed'
GROUP BY src_ip
ORDER BY failed_count DESC;
"""

con.sql(day1_count_query).df()


In [ ]:
day1_enrich_query = f"""
SELECT
    e.timestamp,
    e.src_ip,
    e.user_id,
    e.event_type,
    e.outcome,
    u.department,
    u.role_title,
    u.privileged_account,
    u.sensitive_account
FROM read_csv_auto('{DAY1_EVENTS}') AS e
LEFT JOIN read_csv_auto('{DAY1_USERS}') AS u
    ON e.user_id = u.user_id
WHERE e.src_ip = '198.51.100.77'
ORDER BY e.timestamp;
"""

con.sql(day1_enrich_query).df()


## Day 2 with Pandas

Goal:
- join the alert queue to asset context
- compute the lab's scoring rubric
- rank alerts
- inspect the top escalation candidate


In [ ]:
alerts = pd.read_csv(DAY2_ALERTS)
assets = pd.read_csv(DAY2_ASSETS)

severity_map = {'high': 3, 'medium': 2, 'low': 1}
confidence_map = {'high': 3, 'medium': 2, 'low': 1}
criticality_map = {'high': 3, 'medium': 2, 'low': 1}

merged = alerts.merge(assets, on='asset_id', how='left')
merged['score'] = (
    merged['severity'].map(severity_map)
    + merged['confidence'].map(confidence_map)
    + merged['asset_criticality'].map(criticality_map)
    + (merged['internet_exposed'] == 'yes').astype(int)
    + (merged['sensitive_data'] == 'yes').astype(int)
)

merged[['alert_id', 'alert_type', 'asset_name', 'severity', 'confidence', 'asset_criticality', 'internet_exposed', 'sensitive_data', 'score']].sort_values(['score', 'alert_id'], ascending=[False, True])


In [ ]:
top_alert = merged.sort_values(['score', 'alert_id'], ascending=[False, True]).iloc[0]
top_alert


In [ ]:
merged.loc[
    merged['alert_id'].isin(['A1006', 'A1001', 'A1002', 'A1003']),
    ['alert_id', 'alert_type', 'asset_name', 'notes', 'score']
].sort_values(['score', 'alert_id'], ascending=[False, True])


## Day 2 with DuckDB SQL

This reproduces the same scoring logic in SQL.


In [ ]:
day2_score_query = f"""
WITH scored AS (
    SELECT
        a.alert_id,
        a.alert_type,
        a.severity,
        a.confidence,
        x.asset_name,
        x.asset_criticality,
        x.internet_exposed,
        x.sensitive_data,
        a.notes,
        CASE a.severity
            WHEN 'high' THEN 3
            WHEN 'medium' THEN 2
            ELSE 1
        END
        + CASE a.confidence
            WHEN 'high' THEN 3
            WHEN 'medium' THEN 2
            ELSE 1
        END
        + CASE x.asset_criticality
            WHEN 'high' THEN 3
            WHEN 'medium' THEN 2
            ELSE 1
        END
        + CASE WHEN x.internet_exposed = 'yes' THEN 1 ELSE 0 END
        + CASE WHEN x.sensitive_data = 'yes' THEN 1 ELSE 0 END
        AS total_score
    FROM read_csv_auto('{DAY2_ALERTS}') AS a
    LEFT JOIN read_csv_auto('{DAY2_ASSETS}') AS x
        ON a.asset_id = x.asset_id
)
SELECT *
FROM scored
ORDER BY total_score DESC, alert_id;
"""

con.sql(day2_score_query).df()


## Suggested teaching use

Use this notebook if you want to:
- show students that the CSV logic can be expressed in code
- demonstrate how enrichment works in `pandas` and SQL
- model the difference between raw events and context-enriched analysis
- preview how analyst logic transfers into data tooling

This notebook can be used as a visible Week 11 companion for code-based demonstrations and review.
